# Clase 004 — Estructura reproducible de proyecto

**Parte 0 — Prerrequisitos** · cookiecutter-data-science v2.

> 🎯 Pasar de "carpeta con notebooks" a proyecto profesional con separación de datos, código y documentación.

> ⏱️ ~60 min

## 🗺️ La estructura estándar (CCDS v2)

```
mi-proyecto/
├── README.md
├── pyproject.toml          ← deps y metadata
├── Makefile                ← comandos del proyecto
├── data/
│   ├── raw/                ← INMUTABLE. Nunca editar.
│   ├── interim/            ← transformaciones intermedias
│   ├── processed/          ← listo para modelar
│   └── external/           ← datos de terceros
├── notebooks/
│   ├── 0.01-jvp-eda.ipynb  ← convención: <fase>.<n>-<iniciales>-<descripción>
│   └── 1.02-jvp-modelo.ipynb
├── src/
│   └── mi_proyecto/
│       ├── __init__.py
│       ├── data/           ← carga/limpieza
│       ├── features/       ← feature engineering
│       ├── models/         ← entrenamiento, predicción
│       └── visualization/  ← gráficos
├── reports/
│   └── figures/
├── tests/
└── docs/
```

No es dogma — es **convención**. La ventaja: cualquier DS que la conozca sabe dónde buscar.

## ⚙️ Por qué `data/raw` es sagrado

Regla: **nunca modifiques un archivo en `data/raw/`**. Todo procesamiento escribe a `data/interim/` o `data/processed/`.

**Por qué**:
- Si tu pipeline rompe, puedes regenerar todo desde el origen.
- Permite re-ejecutar análisis con datos diferentes (nuevo período, otra fuente).
- Hace explícito el grafo de dependencias (raw → interim → processed).

In [ ]:
# Demo: estructura típica creada con Path (simulación, no genera nada en disco)
from pathlib import Path

estructura = [
    'mi-proyecto/data/raw/',
    'mi-proyecto/data/interim/',
    'mi-proyecto/data/processed/',
    'mi-proyecto/notebooks/0.01-eda.ipynb',
    'mi-proyecto/src/mi_proyecto/__init__.py',
    'mi-proyecto/src/mi_proyecto/features.py',
    'mi-proyecto/pyproject.toml',
    'mi-proyecto/Makefile',
    'mi-proyecto/README.md',
]
for p in estructura:
    icon = '📁' if p.endswith('/') else '📄'
    print(f'{icon} {p}')

## 🐍 Notebooks vs `src/`

**Notebooks** = exploración. Bocetos. Análisis ad-hoc.
**`src/`** = código de producción que se reutiliza.

**Regla práctica**: cuando copias-pegas una función entre 2 notebooks, es momento de moverla a `src/`. Luego desde notebook:

```python
from mi_proyecto.features import limpia_fechas
df = limpia_fechas(df)
```

Para que esto funcione, el paquete debe estar **instalado en modo editable**:

```bash
pip install -e .   # desde la raíz del proyecto, con pyproject.toml
```

## 📦 `pyproject.toml` como fuente de verdad

En 2026, `pyproject.toml` es el estándar (PEP 621). Reemplaza `setup.py`, `setup.cfg`, y deja `requirements.txt` solo para lockfiles.

Ejemplo mínimo:

```toml
[project]
name = "mi-proyecto"
version = "0.1.0"
requires-python = ">=3.12"
dependencies = [
    "numpy>=2.0",
    "pandas>=2.2",
    "matplotlib>=3.8",
    "scikit-learn>=1.4",
]

[project.optional-dependencies]
dev = ["pytest>=8", "ruff>=0.5", "mypy>=1.10"]

[build-system]
requires = ["setuptools>=61"]
build-backend = "setuptools.build_meta"
```

Luego: `pip install -e ".[dev]"` instala todo + tools de desarrollo.

## 🔧 `Makefile` como interfaz humana

```makefile
.PHONY: setup data train test clean

setup:
	python -m venv .venv && . .venv/bin/activate && pip install -e ".[dev]"

data:
	python -m mi_proyecto.data.make_dataset data/raw data/processed

train:
	python -m mi_proyecto.models.train

test:
	pytest tests/ -v

clean:
	rm -rf data/interim/* data/processed/* models/*
```

Ventaja: `make data` es más legible que recordar el comando exacto, y CI puede llamarlo igual que tú.

## 🚩 Olores de proyecto mal estructurado

Si ves alguno de estos, hay deuda técnica:

- `Untitled27.ipynb` ← notebook sin nombre = código que nadie va a leer
- `final_FINAL_v2.py` ← versionado a mano
- `data/customers_20240801_backup.csv` en git ← datos en git, fecha en filename
- 8 notebooks que cargan y limpian el CSV de la misma forma ← función no extraída
- `requirements.txt` con `numpy` (sin versión) ← reproducibilidad rota
- `notebook.ipynb` con celdas vacías y outputs gigantes ← `nbstripout` lo arregla

## ✅ Checklist

- [ ] Sé generar un proyecto con `cookiecutter-data-science`
- [ ] Entiendo por qué `data/raw/` no se modifica
- [ ] Sé importar desde `src/` en mis notebooks
- [ ] Mi proyecto tiene `pyproject.toml`, no `requirements.txt` suelto
- [ ] Reconozco al menos 3 olores de proyectos mal estructurados

## 📝 Homework

Ver `README.md`. Repo CCDS con `data/raw/penguins.csv`, notebook que importa de `src/`, `Makefile` con `make setup` y `make data`.

## 📖 Definiciones y características

**Cookiecutter**

Generador de proyectos a partir de plantillas. Tomas una plantilla (URL de un repo), respondes 3-5 preguntas y obtienes un proyecto con estructura pre-armada. Característica: idempotente — la plantilla no sabe ni le importa el contenido futuro del proyecto.

**CCDS (cookiecutter-data-science)**

Plantilla específica para proyectos de DS, v2 (2023+). Separa `data/raw`, `data/interim`, `data/processed`, `src/`, `notebooks/`, `reports/`, `docs/`. Es **convención**, no dogma.

**Editable install (`pip install -e .`)**

Instala el paquete pero apuntando al código fuente — los cambios se reflejan sin reinstalar. Habilita `from mi_proyecto.features import x` desde notebooks dentro del repo.

**`pyproject.toml`**

Estándar moderno (PEP 621) para metadata + dependencias + config de tools (ruff, mypy, pytest). Reemplaza `setup.py` + `setup.cfg` + `requirements.txt` suelto + configs sueltas.

**`Makefile`**

Archivo con "recetas" nombradas (`make data`, `make train`). Escrito en tabs (no espacios), las dependencias se declaran arriba (`target: dep1 dep2`). En proyectos DS funciona como interfaz humana a comandos típicos.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `ModuleNotFoundError: No module named 'mi_proyecto'` al importar desde notebook | El paquete no está instalado en el venv del kernel. **Fix**: con el venv activo, `pip install -e .` desde la raíz del proyecto (necesita `pyproject.toml` con `[project] name='mi_proyecto'`). |
| CSV en `data/raw/` apareció en `git status` y pesa 200 MB | El `.gitignore` no cubre `data/raw/*` o no se aplicó a tiempo. **Fix**: añade `data/raw/*` al `.gitignore` + `!data/raw/.gitkeep` (mantén placeholder); si ya commiteaste, `git rm --cached data/raw/customers.csv`. |
| Tengo 3 notebooks con la misma función `limpiar_fechas` | Copy-paste. **Fix**: extrae a `src/mi_proyecto/features/cleaning.py` y haz `from mi_proyecto.features.cleaning import limpiar_fechas` en cada notebook. Si modificas la función, todos los notebooks la heredan. |
| El compañero clonó el repo y `make data` falla con "command not found" | Make no está instalado en Windows por default. **Fix**: instalar make (Git Bash trae uno) o documentar el comando equivalente en README; alternativamente, usa scripts Python directos. |
| Edité `pyproject.toml` y los imports siguen viejos | Tras cambios en metadata o entry-points debes reinstalar. **Fix**: `pip install -e . --force-reinstall --no-deps` (sin deps si no las cambiaste). |

## ❓ Preguntas frecuentes

**❓ ¿Realmente necesito una plantilla? ¿No puedo improvisar?**

Puedes, pero perderás el 80% de la ganancia: el compañero que ya conoce CCDS sabe dónde buscar; sin plantilla, cada proyecto es una caja de sorpresas.

**❓ ¿`data/raw/` o `data/01_raw/`?**

CCDS v2 usa `data/raw/`, `data/interim/`, `data/processed/`, `data/external/`. La numeración `01_/02_/03_` la verás en Kedro y en algunas variantes — ambas son válidas, sigue una sola convención por proyecto.

**❓ ¿`requirements.txt` o `pyproject.toml`?**

**`pyproject.toml`** para declarar las deps de tu paquete. **`requirements.txt`** (generado con `pip freeze` o `uv pip compile`) es el **lockfile** con versiones exactas para reproducir. No están en conflicto; conviven.

**❓ ¿Y si el proyecto es solo un notebook exploratorio?**

No fuerces CCDS. Carpeta con `notebook.ipynb`, `data.csv` y `README.md` está bien. La estructura completa aporta cuando el proyecto vive >3 meses o tiene >1 persona.

**❓ ¿Por qué los notebooks tienen prefijo numérico como `0.01-jvp-eda.ipynb`?**

Convención CCDS: `<fase>.<orden>-<iniciales>-<tema>`. Fase 0=exploración, 1=features, 2=modelos, 3=reportes. Iniciales del autor evitan conflictos cuando varias personas crean notebooks.

## 🔗 Referencias

- [cookiecutter-data-science v2](https://cookiecutter-data-science.drivendata.org/)
- Sculley et al., *Hidden Technical Debt in ML Systems* (NeurIPS 2015)

➡️ **Siguiente:** [005 — VS Code / Cursor para Python y Jupyter](../005-vs-code-cursor-para-python-y-jupyter/README.md)

## 🧭 Walkthrough ejecutable — generá la estructura DS con tus manos

`cookiecutter` necesita internet y responder prompts, así que acá **generamos el árbol típico de un proyecto
de data science** (estilo CCDS v2) con `pathlib` en una carpeta temporal, colocamos `.gitkeep` en las carpetas
que deben existir aunque estén vacías, y luego **recorremos el árbol** para que veas la estructura creada.

En tu terminal real lo generarías con:
`pipx run cookiecutter https://github.com/drivendataorg/cookiecutter-data-science`.

In [ ]:
import tempfile
from pathlib import Path

# Carpetas canonicas de un proyecto DS (CCDS v2)
DIRS = [
    'data/raw', 'data/interim', 'data/processed', 'data/external',
    'notebooks', 'src/mi_proyecto', 'models', 'reports/figures',
    'references', 'docs', 'tests',
]
FILES = {
    'README.md': '# mi_proyecto\n\nProyecto DS generado con estructura CCDS.\n',
    'pyproject.toml': '[project]\nname = "mi_proyecto"\nversion = "0.1.0"\n',
    'Makefile': 'setup:\n\tpip install -e .\n',
    '.gitignore': 'data/raw/*\n!data/raw/.gitkeep\n.venv/\n__pycache__/\n',
    'src/mi_proyecto/__init__.py': '',
}

def build_tree(root: Path):
    for d in DIRS:
        (root / d).mkdir(parents=True, exist_ok=True)
        # .gitkeep para versionar carpetas vacias (git no trackea dirs vacios)
        if not any((root / d).iterdir()):
            (root / d / '.gitkeep').write_text('', encoding='utf-8')
    for rel, content in FILES.items():
        (root / rel).parent.mkdir(parents=True, exist_ok=True)
        (root / rel).write_text(content, encoding='utf-8')

def print_tree(root: Path):
    entradas = sorted(root.rglob('*'), key=lambda x: str(x).lower())
    print(root.name + '/')
    for x in entradas:
        rel = x.relative_to(root)
        indent = '    ' * (len(rel.parts) - 1)
        print(f'{indent}{x.name}' + ('/' if x.is_dir() else ''))

with tempfile.TemporaryDirectory() as tmp:
    proyecto = Path(tmp) / 'ds-lab-004'
    proyecto.mkdir()
    build_tree(proyecto)
    print_tree(proyecto)

    n_dirs = sum(1 for x in proyecto.rglob('*') if x.is_dir())
    print(f'\nCarpetas creadas: {n_dirs}')
    assert (proyecto / 'data' / 'raw' / '.gitkeep').exists()
    assert (proyecto / 'src' / 'mi_proyecto' / '__init__.py').exists()

print('\nOK: separas datos (raw/interim/processed), codigo (src/), exploracion (notebooks/) y salidas (reports/).')

## ✅ Soluciones de los ejercicios

Intentá resolverlos vos primero; acá tenés una solución de referencia comentada. Las adaptamos para que
corran **sin internet** dentro del notebook y demuestren el mismo concepto que harías con la plantilla real.

**Ejercicio 1.** Generá un proyecto CCDS y explorá su estructura. (Adaptación: creamos y verificamos las
carpetas canónicas de CCDS v2, que es lo que la plantilla produce.)

In [ ]:
import tempfile
from pathlib import Path

CCDS = ['data/raw', 'data/interim', 'data/processed', 'data/external',
        'docs', 'models', 'notebooks', 'references', 'reports/figures',
        'src', 'tests']

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp) / 'ds-lab-004'
    for d in CCDS:
        (root / d).mkdir(parents=True, exist_ok=True)
    existentes = sorted(str(Path(d)) for d in CCDS if (root / d).is_dir())
    print('Estructura CCDS v2 generada:')
    for d in existentes:
        print('   ', d)
    assert (root / 'data' / 'raw').is_dir()
    assert (root / 'src').is_dir() and (root / 'notebooks').is_dir()

print('\nOK: en tu maquina  pipx run cookiecutter <url-ccds>  genera esto respondiendo 3-5 preguntas.')

**Ejercicio 2.** Mové una función de un notebook a `src/` e importala. (Creamos un paquete real en `src/`,
lo hacemos importable y lo usamos — como haría un `pip install -e .`.)

In [ ]:
import tempfile, sys, importlib
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    pkg = root / 'src' / 'mi_proyecto'
    pkg.mkdir(parents=True)
    (pkg / '__init__.py').write_text('', encoding='utf-8')
    # La funcion que antes vivia suelta en una celda del notebook, ahora en src/
    (pkg / 'features.py').write_text(
        'def normalize(xs):\n'
        '    m = sum(xs) / len(xs)\n'
        '    var = sum((x - m) ** 2 for x in xs) / len(xs)\n'
        '    sd = var ** 0.5\n'
        '    return [(x - m) / sd for x in xs]\n',
        encoding='utf-8')

    # Hacemos src/ importable (equivalente a  pip install -e .  con pythonpath=src)
    sys.path.insert(0, str(root / 'src'))
    try:
        features = importlib.import_module('mi_proyecto.features')
        out = features.normalize([1.0, 2.0, 3.0, 4.0, 5.0])
        print('normalize([1..5]) ->', [round(v, 3) for v in out])
        assert abs(sum(out) / len(out)) < 1e-9, 'La media normalizada debe ser ~0'
    finally:
        sys.path.remove(str(root / 'src'))
        sys.modules.pop('mi_proyecto.features', None)
        sys.modules.pop('mi_proyecto', None)

print('\nOK: el codigo reusable vive en src/ y se importa con  from mi_proyecto.features import normalize.')

**Ejercicio 3.** Convertí un `requirements.txt` a `pyproject.toml` (sección `[project].dependencies`) y verificá con `tomllib`.

In [ ]:
import tomllib   # stdlib desde Python 3.11

requirements_txt = '''numpy>=2.0
pandas>=2.2
matplotlib>=3.8
'''

deps = [ln.strip() for ln in requirements_txt.splitlines() if ln.strip()]
# Generamos el pyproject.toml
lista_toml = ',\n'.join(f'    "{d}"' for d in deps)
pyproject = (
    '[project]\n'
    'name = "mi_proyecto"\n'
    'version = "0.1.0"\n'
    'requires-python = ">=3.12"\n'
    'dependencies = [\n' + lista_toml + ',\n]\n'
)
print('pyproject.toml generado:\n')
print(pyproject)

# Verificamos parseando de vuelta
parsed = tomllib.loads(pyproject)
print('dependencies parseadas:', parsed['project']['dependencies'])
assert parsed['project']['dependencies'] == deps, 'Las deps deben sobrevivir el round-trip txt -> toml'
print('\nOK: pyproject.toml declara las deps del paquete; requirements.txt queda como lockfile (pip freeze).')

**Ejercicio 4.** Refactorizá un notebook caótico: extraé la lógica copy-paste a **una** función reutilizable.

In [ ]:
# ANTES: dos bloques copy-paste que hacen lo mismo (normalizar) con codigo duplicado
serie_a = [10, 20, 30, 40]
m_a = sum(serie_a) / len(serie_a)
norm_a = [(x - m_a) / (max(serie_a) - min(serie_a)) for x in serie_a]

serie_b = [5, 15, 25, 35]
m_b = sum(serie_b) / len(serie_b)
norm_b = [(x - m_b) / (max(serie_b) - min(serie_b)) for x in serie_b]

# DESPUES: una sola funcion (esto iria a src/mi_proyecto/features.py)
def escala_centrada(xs):
    m = sum(xs) / len(xs)
    rango = max(xs) - min(xs)
    return [(x - m) / rango for x in xs]

# Mismo resultado, sin duplicacion
assert escala_centrada(serie_a) == norm_a
assert escala_centrada(serie_b) == norm_b
print('escala_centrada(serie_a) =', [round(v, 3) for v in escala_centrada(serie_a)])
print('escala_centrada(serie_b) =', [round(v, 3) for v in escala_centrada(serie_b)])
print('\nOK: una funcion en src/ reemplaza N copias; si la mejoras, todos los notebooks la heredan.')

**Ejercicio 5.** Listá 5 olores de un proyecto mal estructurado y su arreglo. (Los representamos como datos verificables.)

In [ ]:
olores = {
    'Untitled27.ipynb / notebooks sin nombrar':
        'Renombrar a  <fase>.<orden>-<iniciales>-<tema>.ipynb  (ej. 0.01-va-eda.ipynb).',
    'Datos pesados versionados en git (data/raw/*.csv de 200 MB)':
        'Ignorar con .gitignore + .gitkeep; datos grandes a DVC / S3 / almacenamiento externo.',
    'La misma funcion copy-pasteada en 3 notebooks':
        'Extraer a src/mi_proyecto/features.py e importar; una sola fuente de verdad.',
    'final_FINAL_v2.py / versionado manual por nombre de archivo':
        'Usar git (commits + tags) en vez de sufijos; el historial ya es tu control de versiones.',
    'Dependencias sin declarar (funciona en mi maquina)':
        'Declararlas en pyproject.toml y congelar un lockfile con  pip freeze / uv pip compile.',
}

for i, (olor, arreglo) in enumerate(olores.items(), 1):
    print(f'{i}. OLOR : {olor}')
    print(f'   FIX  : {arreglo}\n')

assert len(olores) == 5, 'Se piden exactamente 5 olores'
print('OK: reconocer estos olores temprano evita deuda tecnica (Sculley et al., 2015).')